# MedGemma Activation Capture — Breast Cancer PNGs

**Purpose:** Run MedGemma 4B-IT on each breast cancer PNG and capture the exact
hidden state activations at layer-0 attention (q/k/v/o_proj inputs and outputs).

These arrays are then downloaded and fed into the PhotoMedGemma local simulation
(`scripts/compare_with_medgemma.py`) for the paper comparison.

## Setup checklist
1. **Kaggle secret:** Add-ons → Secrets → `HF_TOKEN` (HuggingFace token)
2. **Model access:** Accept terms at https://huggingface.co/google/medgemma-4b-it
3. **Accelerator:** Session → GPU T4 x2 (or any GPU)
4. **Dataset:** Upload 5 breast cancer PNGs as a Kaggle dataset (any name)
   - `10025_893612858.png`, `10011_1031443799.png`, `10042_102733848.png`,
     `10042_1648588715.png`, `10042_495770405.png`

## What to download after running
From the Kaggle output tab, download everything in `/kaggle/working/photomedgemma_outputs/`:
- `W_q_layer0.npy`, `W_k_layer0.npy`, `W_v_layer0.npy`, `W_o_layer0.npy`
- `input_layer0_*_<stem>.npy` (hidden states into each projection)
- `output_layer0_*_<stem>.npy` (MedGemma outputs from each projection)

In [ ]:
!pip install -q transformers accelerate bitsandbytes Pillow numpy

In [ ]:
import os, json, glob, traceback
import numpy as np
from pathlib import Path

MODEL_ID      = 'google/medgemma-4b-it'
QUANTIZE_4BIT = True   # True = ~3.5 GB VRAM (T4 ok); False = ~8 GB (A100)
QUERY         = 'Describe any abnormalities visible in this medical image in detail.'
OUTPUT_DIR    = Path('/kaggle/working/photomedgemma_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

# HuggingFace token
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token: loaded from Kaggle secrets')
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('huggingFace_token')
        print('HF token: loaded (huggingFace_token key)')
    except Exception:
        HF_TOKEN = os.environ.get('HF_TOKEN', '')
        print(f'HF token: env ({bool(HF_TOKEN)})')

# Find images — search all of /kaggle/input/ recursively (works for public
# datasets, private datasets, and any folder name the user chose)
image_paths = sorted(glob.glob('/kaggle/input/**/*.png', recursive=True))
if not image_paths:
    image_paths = sorted(glob.glob('/kaggle/working/*.png'))
if not image_paths:
    print('WARNING: No PNG images found. Check that your dataset is attached.')

print(f'Model:  {MODEL_ID}')
print(f'Quant:  {"4-bit NF4" if QUANTIZE_4BIT else "bfloat16"}')
print(f'Images: {len(image_paths)}')
for p in image_paths:
    print(f'  {p}')

## 1. Load MedGemma

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

if torch.cuda.is_available():
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('No GPU — running on CPU (very slow)')

processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)

if QUANTIZE_4BIT:
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, token=HF_TOKEN,
        quantization_config=bnb_cfg,
        device_map='auto',
    )
else:
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, token=HF_TOKEN,
        torch_dtype=torch.bfloat16,
        device_map='auto',
    )

model.eval()
print(f'Model class: {type(model).__name__}')
print(f'Params:      {sum(p.numel() for p in model.parameters())/1e9:.2f}B')

## 2. Auto-discover layer-0 attention projections

In [ ]:
def _get_transformer_layers(m):
    """Return the list of transformer decoder layers, regardless of HF class."""
    probes = [
        lambda m: m.model.language_model.layers,   # Gemma3ForConditionalGeneration (confirmed)
        lambda m: m.model.layers,
        lambda m: m.language_model.model.layers,
        lambda m: m.model.language_model.model.layers,
        lambda m: m.model.decoder.layers,
        lambda m: m.transformer.h,
    ]
    for fn in probes:
        try:
            layers = fn(m)
            if layers is not None and len(layers) > 0:
                return layers
        except AttributeError:
            pass
    raise RuntimeError(
        f'Cannot find transformer layers in {type(m).__name__}.\n'
        f'Top-level children: {[n for n, _ in m.named_children()]}'
    )

def _get_attn_proj(layer, proj_name):
    """Get q/k/v/o_proj from a decoder layer."""
    for attn_attr in ('self_attn', 'attention', 'attn'):
        attn = getattr(layer, attn_attr, None)
        if attn is not None:
            proj = getattr(attn, proj_name, None)
            if proj is not None:
                return proj
    raise AttributeError(f'{proj_name} not found in layer {type(layer).__name__}')

transformer_layers = _get_transformer_layers(model)
layer0 = transformer_layers[0]

print(f'Transformer layers: {len(transformer_layers)}')
print(f'Layer 0 type:       {type(layer0).__name__}')
for pn in ('q_proj', 'k_proj', 'v_proj', 'o_proj'):
    proj = _get_attn_proj(layer0, pn)
    print(f'  {pn}: weight={tuple(proj.weight.shape)}')

## 3. Extract layer-0 weight matrices and save

In [ ]:
def extract_weight(layer_idx, proj_name):
    """Extract weight as float32 numpy. Handles NF4 dequantization."""
    layer = transformer_layers[layer_idx]
    proj  = _get_attn_proj(layer, proj_name)
    w = proj.weight
    if QUANTIZE_4BIT and hasattr(w, 'quant_state'):
        try:
            import bitsandbytes as bnb
            return bnb.functional.dequantize_4bit(
                w.data, w.quant_state
            ).float().cpu().numpy()
        except Exception as e:
            print(f'  bitsandbytes dequant failed ({e}), using .data fallback')
    return w.detach().float().cpu().numpy()

W_q = extract_weight(0, 'q_proj')
W_k = extract_weight(0, 'k_proj')
W_v = extract_weight(0, 'v_proj')
W_o = extract_weight(0, 'o_proj')

print('Layer-0 weights:')
for name, W in [('q_proj', W_q), ('k_proj', W_k), ('v_proj', W_v), ('o_proj', W_o)]:
    print(f'  {name}: {W.shape}  norm={np.linalg.norm(W):.3f}')

# Save — needed by compare_with_medgemma.py locally
for tag, W in [('W_q', W_q), ('W_k', W_k), ('W_v', W_v), ('W_o', W_o)]:
    np.save(OUTPUT_DIR / f'{tag}_layer0.npy', W)
print('Weights saved.')

## 4. Run MedGemma on each image — capture layer-0 activations

In [ ]:
from PIL import Image

def run_image(image_path):
    """
    Run MedGemma on one PNG:
      - Registers hooks on layer-0 q/k/v/o_proj to capture input hidden states
        and output activations
      - Generates a text response
      - Saves input and output arrays as .npy files
    """
    pil_img = Image.open(image_path).convert('RGB')
    stem    = Path(image_path).stem
    print(f'\n{"="*60}')
    print(f'Image: {Path(image_path).name}  ({pil_img.height}x{pil_img.width})')

    # Register forward hooks on layer-0 projections
    captured = {}
    hooks    = []

    def _hook(name):
        def fn(module, inp, out):
            tensor = out[0] if isinstance(out, (tuple, list)) else out
            captured[name] = {
                'input':  inp[0].detach().float().cpu().numpy(),
                'output': tensor.detach().float().cpu().numpy(),
            }
        return fn

    layer0 = transformer_layers[0]
    for pname in ('q_proj', 'k_proj', 'v_proj', 'o_proj'):
        proj = _get_attn_proj(layer0, pname)
        hooks.append(proj.register_forward_hook(_hook(pname)))

    # Build model inputs
    messages = [{'role': 'user', 'content': [
        {'type': 'image'},
        {'type': 'text', 'text': QUERY},
    ]}]
    try:
        prompt = processor.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False
        )
        inputs = processor(text=prompt, images=[pil_img], return_tensors='pt')
    except Exception:
        inputs = processor.apply_chat_template(
            messages, add_generation_prompt=True,
            tokenize=True, return_dict=True, return_tensors='pt',
        )

    first_device = next(model.parameters()).device
    inputs = {k: v.to(first_device) if hasattr(v, 'to') else v
              for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=150, do_sample=False)

    for h in hooks:
        h.remove()

    n_in     = inputs['input_ids'].shape[1]
    response = processor.decode(output_ids[0, n_in:], skip_special_tokens=True)
    print(f'Tokens: {n_in}   Captured: {list(captured.keys())}')
    print(f'Response: {response[:200]}...')

    # Save input and output arrays for each projection
    saved_files = []
    for pname, data in captured.items():
        inp = data['input']
        out = data['output']
        # Remove batch dim
        if inp.ndim == 3: inp = inp[0]
        if out.ndim == 3: out = out[0]
        safe = pname.replace('_', '')
        f_in  = OUTPUT_DIR / f'input_layer0_{safe}_{stem}.npy'
        f_out = OUTPUT_DIR / f'output_layer0_{safe}_{stem}.npy'
        np.save(f_in,  inp)
        np.save(f_out, out)
        saved_files += [f_in.name, f_out.name]
        print(f'  {pname}: input={inp.shape}  output={out.shape}')

    result = {
        'image':          Path(image_path).name,
        'stem':           stem,
        'n_input_tokens': n_in,
        'response':       response,
        'saved_files':    saved_files,
    }
    (OUTPUT_DIR / f'activations_{stem}.json').write_text(json.dumps(result, indent=2))
    return result


all_results = []
for img_path in image_paths:
    try:
        all_results.append(run_image(img_path))
    except Exception as e:
        print(f'ERROR on {img_path}: {e}')
        traceback.print_exc()

print(f'\nCompleted {len(all_results)}/{len(image_paths)} images.')

## 5. Summary — download these files then run the local comparison

In [ ]:
SEP = '=' * 70
print(SEP)
print('MedGemma Activation Capture — Complete')
print(SEP)
print()
print('MedGemma responses:')
print('-' * 70)
for r in all_results:
    print(f"\n[{r['stem']}]")
    for line in r['response'].split('\n')[:4]:
        print(f'  {line}')

print()
print('Files to download from Kaggle output tab:')
print('-' * 70)
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {f.name:<60} {f.stat().st_size/1024:>7.0f} KB')

print()
print('After downloading, run the local comparison in VSCode:')
print()
print('  mkdir -p output/simulations/kaggle_activations')
print('  cp ~/Downloads/W_*_layer0.npy           output/simulations/kaggle_activations/')
print('  cp ~/Downloads/input_layer0_*.npy        output/simulations/kaggle_activations/')
print('  cp ~/Downloads/output_layer0_*.npy       output/simulations/kaggle_activations/')
print()
print('  python3 scripts/compare_with_medgemma.py \\')
print('    --activations-dir output/simulations/kaggle_activations/ \\')
print('    --phase-map       output/real/phase_map.json')
print(SEP)